In [0]:
%sql
select ChannelName, Title, PublishedAt, v.ViewCount, v.LikeCount, v.CommentCount, v.ScrapedAt
from youtube_lakehouse.silver.fact_videos_history v
inner join (select distinct channelID, channelName from youtube_lakehouse.silver.dim_channels_history ) c
on v.ChannelID = c.ChannelID
order by PublishedAt desc


In [0]:
select channel_title, video_title, published_at, cumulative_views, cumulative_likes, cumulative_comments, snapshot_date
from youtube_lakehouse.silver.fact_video_daily_snapshots
order by published_at desc

In [0]:
select ChannelName, Title, PublishedAt, v.ViewCount, v.LikeCount, v.CommentCount, v.ScrapedAt
from youtube_lakehouse.silver.fact_videos_history v
inner join (select distinct channelID, channelName from youtube_lakehouse.silver.dim_channels_history ) c
on v.ChannelID = c.ChannelID
--where ChannelName = 'WildLens by Abrar'
union all
select channel_title, video_title, published_at, cumulative_views, cumulative_likes, cumulative_comments, snapshot_date
from youtube_lakehouse.silver.fact_video_daily_snapshots
--where channel_title = 'WildLens by Abrar'
order by channelname, PublishedAt,  v.ScrapedAt;


In [0]:
with daily_snapshot as (select distinct  Title, 
    --PublishedAt, 
    max(v.ViewCount) over(partition by Title, date(v.ScrapedAt)) as maxViews, 
    --max(v.LikeCount) over(partition by Title, date(v.ScrapedAt)) as maxLikeCount, 
    --max(v.CommentCount) over(partition by Title, date(v.ScrapedAt)) as maxCommentCount, 
    date(v.ScrapedAt)
from youtube_lakehouse.silver.fact_videos_history v
where ChannelID = 'UCf1XIplYiqNv9baGsFHwPPQ'),
daily_deltas as (select *, 
    maxViews - lag(v.maxViews,1,0) over(partition by Title order by date(v.ScrapedAt)) as DoDViews_Delta,
    row_number() over(partition by Title order by date(v.ScrapedAt)) as DoDViews_Delta_Rank
from daily_snapshot v)
select * from daily_deltas
where dodviews_delta_rank > 1
order by Title, ScrapedAt;


In [0]:
with daily_snapshot as (
  select 
    ChannelID,
    Title,
    videoid,
    date(publishedAt),
    date(ScrapedAt) as snapshot_date,
    max(ViewCount) as maxViews,
    max(LikeCount) as maxLikes,
    max(CommentCount) as maxComment
  from youtube_lakehouse.silver.fact_videos_history
  --where ChannelID = 'UCf1XIplYiqNv9baGsFHwPPQ'
  group by Title, ChannelID, videoid,  date(publishedAt),date(ScrapedAt)
),
daily_deltas as (
  select 
    Title,
    publishedAt,
    snapshot_date,
    maxViews,
    maxViews - lag(maxViews, 1) over (
      partition by Title 
      order by snapshot_date
    ) as DoDViews_Delta,
    maxLikes,
    maxLikes - lag(maxLikes, 1) over (
      partition by Title 
      order by snapshot_date
    ) as DoDLikes_Delta,
    maxComment,
    maxComment - lag(maxComment, 1) over (
      partition by Title 
      order by snapshot_date
    ) as DoDComment_Delta
  from daily_snapshot
)
select 
  Title,
  publishedAt,
  snapshot_date,
  maxViews,
  DoDViews_Delta,
  DoDLikes_Delta,
  DoDComment_Delta
from daily_deltas
where   publishedat > date('2026-08-29')
order by publishedAt, Title, snapshot_date;

In [0]:
select ChannelName,c.channelid, Title, PublishedAt, max(v.ViewCount) over(partition by ChannelName, Title order by ScrapedAt) as maxview , v.LikeCount, v.CommentCount, v.ScrapedAt
from youtube_lakehouse.silver.fact_videos_history v
inner join (select distinct channelID, channelName from youtube_lakehouse.silver.dim_channels_history ) c
on v.ChannelID = c.ChannelID
where ChannelName = 'WildLens by Abrar'
order by VideoID, ScrapedAt